# NCL + SAE Brain MRI — SIMCLR Training Notebook
Ye notebook sirf **simclr** backbone train karti hai. Sab cells top-se-bottom run karo — koi variable change nahi karni.

Dataset, model, training loop, aur results-saving sab isi notebook mein hain (self-contained).

## 1. Imports + Config

In [ ]:
import os, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
import timm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ==================== CONFIG (apne paths se update karo) ====================
METADATA_CSV = r"C:\Users\Raaghav\Desktop\coding\research\brain-mri-ncl-simclr\data\processed_flair\metadata.csv"
CHECKPOINT_DIR = r"C:\Users\Raaghav\Desktop\coding\research\brain-mri-ncl-simclr\checkpoints"
RESULTS_DIR = r"C:\Users\Raaghav\Desktop\coding\research\brain-mri-ncl-simclr\results"
IMG_SIZE = 128
BATCH_SIZE = 64        # OOM aaye toh 32 kar do
NUM_WORKERS = 0        # Windows + notebook mein 0 safest hai
LR = 3e-4
TEMPERATURE = 0.5
SEED = 42              # paper reproducibility ke liye fix rakho, mat badlo
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "", "| Seed:", SEED)


## 2. Dataset classes

In [ ]:
class BraTSFlairDataset(Dataset):
    """Supervised training ke liye — single view + label."""
    def __init__(self, metadata_csv, split, augment=False):
        df = pd.read_csv(metadata_csv)
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.transform = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10),
        ]) if augment else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        data = np.load(row["file_path"], allow_pickle=True).item()
        image = torch.from_numpy(data["image"]).unsqueeze(0).float()
        if self.transform:
            image = self.transform(image)
        return image, int(row["has_tumor"])


class BraTSContrastiveDataset(Dataset):
    """SimCLR/NCL ke liye — do augmented views."""
    def __init__(self, metadata_csv, split):
        df = pd.read_csv(metadata_csv)
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.transform = T.Compose([
            T.RandomResizedCrop(size=IMG_SIZE, scale=(0.7, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=15),
            T.GaussianNoise(sigma=0.03),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        data = np.load(row["file_path"], allow_pickle=True).item()
        base = torch.from_numpy(data["image"]).unsqueeze(0).float()
        return self.transform(base), self.transform(base)

print("Dataset classes ready.")

## 2b. Sanity check — ek batch visualize karo

In [ ]:
_check_ds = BraTSFlairDataset(METADATA_CSV, split="train")
print(f"Train samples: {len(_check_ds)}")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    img, label = _check_ds[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"tumor={label}")
    ax.axis("off")
plt.show()

## 3. Model classes

In [ ]:
def build_vit_tiny_backbone(img_size=IMG_SIZE, in_chans=1):
    return timm.create_model(
        "vit_tiny_patch16_224", pretrained=False,
        img_size=img_size, in_chans=in_chans, num_classes=0,
    )

class ProjectionHead(nn.Module):
    """non_negative=True => NCL (bas ek extra ReLU, yahi SimCLR se fark hai)."""
    def __init__(self, in_dim=192, hidden_dim=192, out_dim=128, non_negative=False):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.ReLU(inplace=True), nn.Linear(hidden_dim, out_dim)]
        if non_negative:
            layers.append(nn.ReLU(inplace=True))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class SupervisedViT(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = build_vit_tiny_backbone()
        self.head = nn.Linear(192, num_classes)
    def forward(self, x):
        return self.head(self.backbone(x))

class ContrastiveViT(nn.Module):
    def __init__(self, non_negative=False):
        super().__init__()
        self.backbone = build_vit_tiny_backbone()
        self.projector = ProjectionHead(non_negative=non_negative)
    def forward(self, x):
        feats = self.backbone(x)
        return feats, self.projector(feats)

print("Model classes ready.")

## 4. Loss + checkpoint helper

In [ ]:
def nt_xent_loss(z1, z2, temperature=TEMPERATURE):
    batch_size = z1.size(0)
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    reps = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(reps, reps.T) / temperature
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z1.device)
    sim.masked_fill_(mask, float("-inf"))
    pos_idx = torch.cat([torch.arange(batch_size, 2*batch_size), torch.arange(0, batch_size)]).to(z1.device)
    return F.cross_entropy(sim, pos_idx)


def _atomic_write_json(path, obj):
    """Crash-safe JSON write: temp file pe likho, phir atomic rename. Corruption se bachata hai."""
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp_path, path)  # atomic on same filesystem


def save_checkpoint(backbone, mode_name, epoch, extra_state=None, is_best=False, metric=None):
    """Har epoch 'latest' checkpoint save karta hai (crash-recovery ke liye),
    aur agar is_best=True toh alag 'best' checkpoint bhi save karta hai (overwrite se safe)."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    payload = {"backbone_state_dict": backbone.state_dict(), "epoch": epoch, "seed": SEED}
    if metric is not None:
        payload["metric"] = metric
    if extra_state:
        payload.update(extra_state)

    latest_path = os.path.join(CHECKPOINT_DIR, f"{mode_name}_backbone_latest.pt")
    tmp_path = latest_path + ".tmp"
    try:
        torch.save(payload, tmp_path)
        os.replace(tmp_path, latest_path)  # atomic swap, torch.save beech mein fail ho toh old file safe rehti hai
    except Exception as e:
        print(f"[WARN] checkpoint save failed at epoch {epoch}: {e}")
        return

    if is_best:
        best_path = os.path.join(CHECKPOINT_DIR, f"{mode_name}_backbone_best.pt")
        try:
            torch.save(payload, best_path + ".tmp")
            os.replace(best_path + ".tmp", best_path)
        except Exception as e:
            print(f"[WARN] best-checkpoint save failed at epoch {epoch}: {e}")


import json

def save_config(mode_name):
    """Reproducibility ke liye run ka config JSON mein snapshot karta hai."""
    os.makedirs(RESULTS_DIR, exist_ok=True)
    cfg = {
        "mode": mode_name, "seed": SEED, "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
        "lr": LR, "temperature": TEMPERATURE if mode_name != "supervised" else None,
        "device": DEVICE, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    _atomic_write_json(os.path.join(RESULTS_DIR, f"{mode_name}_config.json"), cfg)


def save_results(mode_name, history, fig):
    """History (JSON) + final loss curve (PNG) + running summary.json update karta hai. Crash-safe."""
    os.makedirs(RESULTS_DIR, exist_ok=True)
    figures_dir = os.path.join(RESULTS_DIR, "figures")
    os.makedirs(figures_dir, exist_ok=True)

    # 1. Full per-epoch history (atomic write)
    history_path = os.path.join(RESULTS_DIR, f"{mode_name}_history.json")
    try:
        _atomic_write_json(history_path, history)
    except Exception as e:
        print(f"[WARN] history save failed: {e}")

    # 2. Final loss/accuracy curve figure (paper-ready PNG)
    fig_path = os.path.join(figures_dir, f"{mode_name}_training_curve.png")
    if fig is not None:
        try:
            fig.savefig(fig_path, dpi=150, bbox_inches="tight")
        except Exception as e:
            print(f"[WARN] figure save failed: {e}")

    # 3. Running summary.json (sab modes ka final metrics ek jagah), atomic read-modify-write
    summary_path = os.path.join(RESULTS_DIR, "summary.json")
    summary = {}
    if os.path.exists(summary_path):
        try:
            with open(summary_path) as f:
                summary = json.load(f)
        except json.JSONDecodeError:
            print("[WARN] existing summary.json corrupt, starting fresh (old file untouched on disk as backup)")
            os.replace(summary_path, summary_path + ".corrupt_backup")
            summary = {}

    entry = {
        "final_train_loss": history["train_loss"][-1],
        "total_epochs": len(history["train_loss"]),
        "total_time_min": sum(history["epoch_times_min"]),
        "seed": SEED,
        "checkpoint_latest": os.path.join(CHECKPOINT_DIR, f"{mode_name}_backbone_latest.pt"),
        "checkpoint_best": os.path.join(CHECKPOINT_DIR, f"{mode_name}_backbone_best.pt"),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if "val_acc" in history:
        entry["final_val_acc"] = history["val_acc"][-1]
        entry["best_val_acc"] = max(history["val_acc"])
    if "best_loss" in history:
        entry["best_train_loss"] = history["best_loss"]

    summary[mode_name] = entry
    try:
        _atomic_write_json(summary_path, summary)
    except Exception as e:
        print(f"[WARN] summary.json save failed: {e}")

    print(f"\nResults saved:")
    print(f"  History: {history_path}")
    print(f"  Figure:  {fig_path}")
    print(f"  Summary: {summary_path}")

print("Helpers ready.")


## 5. Training function (live loss plot + progress bar)

In [ ]:
def train_supervised_live(epochs):
    save_config("supervised")
    train_ds = BraTSFlairDataset(METADATA_CSV, "train", augment=True)
    val_ds = BraTSFlairDataset(METADATA_CSV, "val", augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = SupervisedViT().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler(DEVICE)
    criterion = nn.CrossEntropyLoss()

    losses, val_accs, epoch_times = [], [], []
    best_val_acc = -1.0
    fig = None
    for epoch in range(epochs):
        model.train()
        epoch_start = time.time()
        total_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast(DEVICE):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                with torch.amp.autocast(DEVICE):
                    preds = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_loss = total_loss / len(train_loader)
        val_acc = correct / total
        epoch_time_min = (time.time() - epoch_start) / 60
        losses.append(avg_loss)
        val_accs.append(val_acc)
        epoch_times.append(epoch_time_min)

        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
        save_checkpoint(model.backbone, "supervised", epoch, is_best=is_best, metric=val_acc)

        # ---- Live plot update ----
        clear_output(wait=True)
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
        ax[0].plot(losses, marker="o"); ax[0].set_title("Train Loss"); ax[0].set_xlabel("Epoch")
        ax[1].plot(val_accs, marker="o", color="green"); ax[1].set_title("Val Accuracy"); ax[1].set_xlabel("Epoch")
        plt.tight_layout(); plt.show()
        print(f"Epoch {epoch+1}/{epochs} | loss={avg_loss:.4f} | val_acc={val_acc:.4f} | "
              f"best_val_acc={best_val_acc:.4f} | time={epoch_time_min:.1f} min")

        # ---- Intermediate save har epoch (crash-safe: beech mein ruka toh bhi history tak ka data safe) ----
        history_so_far = {"train_loss": losses, "val_acc": val_accs, "epoch_times_min": epoch_times}
        save_results("supervised", history_so_far, fig)
        plt.close(fig)  # memory leak avoid karne ke liye

    history = {"train_loss": losses, "val_acc": val_accs, "epoch_times_min": epoch_times}
    return model, history


def train_contrastive_live(epochs, non_negative):
    mode_name = "ncl" if non_negative else "simclr"
    save_config(mode_name)
    train_ds = BraTSContrastiveDataset(METADATA_CSV, "train")
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)

    model = ContrastiveViT(non_negative=non_negative).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler(DEVICE)

    losses, epoch_times = [], []
    best_loss = float("inf")
    fig = None
    for epoch in range(epochs):
        model.train()
        epoch_start = time.time()
        total_loss = 0.0
        for view1, view2 in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            view1, view2 = view1.to(DEVICE), view2.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast(DEVICE):
                _, z1 = model(view1)
                _, z2 = model(view2)
                loss = nt_xent_loss(z1, z2)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        epoch_time_min = (time.time() - epoch_start) / 60
        losses.append(avg_loss)
        epoch_times.append(epoch_time_min)

        is_best = avg_loss < best_loss
        if is_best:
            best_loss = avg_loss
        save_checkpoint(model.backbone, mode_name, epoch, is_best=is_best, metric=avg_loss)

        clear_output(wait=True)
        fig = plt.figure(figsize=(6, 3.5))
        plt.plot(losses, marker="o")
        plt.title(f"{mode_name.upper()} Contrastive Loss"); plt.xlabel("Epoch")
        plt.tight_layout(); plt.show()
        print(f"[{mode_name.upper()}] Epoch {epoch+1}/{epochs} | loss={avg_loss:.4f} | "
              f"best_loss={best_loss:.4f} | time={epoch_time_min:.1f} min")

        # ---- Intermediate save har epoch (crash-safe) ----
        history_so_far = {"train_loss": losses, "epoch_times_min": epoch_times, "best_loss": best_loss}
        save_results(mode_name, history_so_far, fig)
        plt.close(fig)  # memory leak avoid karne ke liye

    history = {"train_loss": losses, "epoch_times_min": epoch_times, "best_loss": best_loss}
    return model, history

print("Training functions ready.")


## 6. Run training
**Yahan `MODE` badal ke cell dobara run karo** — teeno backbones ke liye (`"supervised"`, `"simclr"`, `"ncl"`). Loss curve live update hoga upar hi, notebook mein.

In [ ]:
MODE = "simclr"
EPOCHS = 25   # pehle EPOCHS=1 se quick test kar sakte ho

model, history = train_contrastive_live(EPOCHS, non_negative=False)

print(f"Done. Checkpoint: {CHECKPOINT_DIR}\\{MODE}_backbone.pt")
print(f"Results: {RESULTS_DIR}\\{MODE}_history.json")